# Classificação — Previsão de Atraso na Saída

Neste notebook, treinaremos **3 modelos de classificação** para prever a variável `ATRASO_SAIDA` (binária: 0 = sem atraso, 1 = com atraso):

1. **XGBoost** (XGBClassifier)
2. **LightGBM** (LGBMClassifier)
3. **Random Forest** (RandomForestClassifier)

Após o treinamento, avaliaremos cada modelo com métricas de classificação e escolheremos o melhor.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    accuracy_score, f1_score, precision_score, recall_score,
    RocCurveDisplay
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings('ignore')

print("Bibliotecas importadas com sucesso.")

## 1. Carregamento dos Dados

In [ ]:
X_train = pd.read_csv('dados/X_train.csv')
X_test = pd.read_csv('dados/X_test.csv')
y_train_full = pd.read_csv('dados/y_train.csv')
y_test_full = pd.read_csv('dados/y_test.csv')

# Selecionar apenas o target ATRASO_SAIDA
y_train = y_train_full['ATRASO_SAIDA']
y_test = y_test_full['ATRASO_SAIDA']

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"\nDistribuição do target (treino):")
print(y_train.value_counts(normalize=True).round(4))
print(f"\nDistribuição do target (teste):")
print(y_test.value_counts(normalize=True).round(4))

## 2. Treinamento dos Modelos

Treinaremos 3 classificadores com os hiperparâmetros padrão e `random_state=42` para reprodutibilidade. Usamos `scale_pos_weight` no XGBoost e `is_unbalanced` no LightGBM para lidar com eventual desbalanceamento, e `class_weight='balanced'` no Random Forest.

In [ ]:
# Calcular peso para classes desbalanceadas
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Razão classe 0 / classe 1: {ratio:.2f}")

# Definir os 3 modelos
modelos = {
    'XGBoost': XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=ratio,
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        is_unbalance=True,
        random_state=42,
        verbose=-1,
        n_jobs=-1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        max_depth=20,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
}

# Treinar cada modelo
for nome, modelo in modelos.items():
    print(f"Treinando {nome}...")
    modelo.fit(X_train, y_train)
    print(f"  ✓ {nome} treinado.")

print("\nTodos os modelos foram treinados!")

## 3. Avaliação dos Modelos

Vamos avaliar cada modelo no conjunto de **teste** usando as seguintes métricas:
- **Accuracy** — proporção de acertos gerais
- **Precision** — dos que o modelo disse "atraso", quantos realmente atrasaram
- **Recall** — dos que realmente atrasaram, quantos o modelo identificou
- **F1-Score** — média harmônica entre precision e recall
- **ROC-AUC** — capacidade de separação entre classes

In [ ]:
resultados = []

for nome, modelo in modelos.items():
    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_proba)
    
    resultados.append({
        'Modelo': nome,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': roc
    })
    
    print(f"\n{'='*50}")
    print(f"  {nome}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=['Sem Atraso', 'Com Atraso']))

df_resultados = pd.DataFrame(resultados).set_index('Modelo')
df_resultados = df_resultados.round(4)
print("\n📊 Resumo Comparativo:")
df_resultados

### 3.1 Comparação Visual das Métricas

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

df_plot = df_resultados.reset_index().melt(id_vars='Modelo', var_name='Métrica', value_name='Valor')
sns.barplot(data=df_plot, x='Métrica', y='Valor', hue='Modelo', ax=ax)
ax.set_ylim(0, 1)
ax.set_title('Comparação de Métricas por Modelo', fontsize=14)
ax.legend(loc='lower right')

for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)

plt.tight_layout()
plt.show()

### 3.2 Matrizes de Confusão

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (nome, modelo) in zip(axes, modelos.items()):
    y_pred = modelo.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Sem Atraso', 'Com Atraso'],
                yticklabels=['Sem Atraso', 'Com Atraso'])
    ax.set_title(f'{nome}', fontsize=13)
    ax.set_xlabel('Predito')
    ax.set_ylabel('Real')

plt.suptitle('Matrizes de Confusão', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### 3.3 Curvas ROC

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for nome, modelo in modelos.items():
    RocCurveDisplay.from_estimator(modelo, X_test, y_test, ax=ax, name=nome)

ax.plot([0, 1], [0, 1], 'k--', label='Aleatório (AUC=0.5)')
ax.set_title('Curvas ROC — Comparação dos Modelos', fontsize=14)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 3.4 Feature Importance — Top 15

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (nome, modelo) in zip(axes, modelos.items()):
    importances = pd.Series(modelo.feature_importances_, index=X_train.columns)
    top15 = importances.nlargest(15)
    top15.sort_values().plot.barh(ax=ax, color='steelblue')
    ax.set_title(f'{nome} — Top 15 Features', fontsize=12)
    ax.set_xlabel('Importância')

plt.tight_layout()
plt.show()

## 4. Seleção do Melhor Modelo

Vamos escolher o modelo com maior **F1-Score**, pois em problemas de atraso é importante balancear tanto a precisão (evitar falsos alarmes) quanto o recall (não deixar atrasos passarem despercebidos).

In [ ]:
melhor_modelo_nome = df_resultados['F1-Score'].idxmax()
melhor_f1 = df_resultados.loc[melhor_modelo_nome, 'F1-Score']
melhor_roc = df_resultados.loc[melhor_modelo_nome, 'ROC-AUC']

print(f"🏆 Melhor modelo: {melhor_modelo_nome}")
print(f"   F1-Score: {melhor_f1:.4f}")
print(f"   ROC-AUC:  {melhor_roc:.4f}")
print(f"\nResumo completo:")
df_resultados.style.highlight_max(axis=0, color='lightgreen')